In [95]:
import os
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

In [ ]:
PROMPT_FILE = "Prompt_Coding_Check_file"

In [ ]:
prompts = pd.read_excel(PROMPT_FILE, sheet_name="Coding")
prompts.head()

In [ ]:
import pandas as pd, re
from collections import Counter
from openpyxl import load_workbook

MASTER_FILE = "All_Application_Findings"   # the master detection matrix
MASTER_SHEET = "Findings"
MAPPING_FILE = "Vulnerability_Mapping"       # the reviewed CWE/OWASP/CVSS mapping
MAPPING_SHEET = "Mapping"
PROMPTS      = "Prompt_File"        # per-team prompt coding (one sheet per team)

# ---- 1. verified vulnerabilities, per app ------------------------------------
df = pd.read_excel(MASTER, sheet_name=MASTER_SHEET)
df.columns = [str(c).strip() for c in df.columns]
df = df[df["Status"].isin(["TP", "FN"])].copy()
df["app"]  = df["Platform"].astype(str).str[0] + "-T" + \
             df["App ID"].astype(str).str.extract(r"Team (\d+)")[0].apply(lambda x: f"{int(x):02d}")
df["cat"]  = df["Standard Category"].astype(str).str.strip()
df["plat"] = df["Platform"]
prev = df.groupby(["cat", "plat"])["app"].nunique().unstack(fill_value=0)



In [99]:
# from the mapping file: only what is genuinely category-level
m = pd.read_excel(MAPPING, sheet_name=MAPPING_SHEET)
m.columns = [str(c).strip() for c in m.columns]
m = m.dropna(subset=["Vulnerability category"]).copy()
m["cat"]   = m["Vulnerability category"].astype(str).str.strip()
owasp_by_cat = dict(zip(m["cat"], m["OWASP Top 10:2025 (from final CWE)"].astype(str).str.extract(r"(A\d\d)")[0]))
sev_by_cat   = dict(zip(m["cat"], m["Proposed severity"].astype(str).str.strip()))

In [ ]:
# ---- 2. origin attribution (computed in the script) ----

PLATFORM_DEFAULT = {
    "Missing browser security headers (CSP, X-Content-Type-Options)",
    "Auth token in localStorage",
    "Deprecated TLS versions", "Mutable search_path in DB functions",
    "No package release-age policy", "Access token valid after logout",
    "Insufficient logging", "Missing server-side input validation",
    "File type validation missing", "Leaked-password check disabled",
}

PROMPT_ASSOCIATED = {
    ("Leaked-password check disabled", "L-T17"):
        "Prompt #16: 'allow all types of password' — weakness explicitly requested (B3).",
}

AI_GAP_WHY = {
    "Lower role bypasses invoice approval": "User specified owner-only approval; app allowed a lower role to approve.",
    "Exposed business and personal data": "User specified role/tenant-scoped access; RLS allowed unauthorised read.",
    "Customer can view all invoices": "User specified customers see only their own invoices; RLS allowed reading all.",
    "Self-service role escalation": "User specified fixed roles; app allowed self-role-change to owner.",
    "Unauthorized file download": "User specified customers see only their own invoices; storage signing not scoped.",
    "Unrestricted invitations": "User specified owner-only invites; app allowed a lower role to invite.",
    "Unauthenticated privileged endpoint": "User asked to seed demo data; AI built a public no-auth endpoint.",
    "Sensitive error messages": "AI generated a recursive RLS policy; raw DB error exposed.",
    "Leaked secrets and credentials": "User supplied test credentials to seed demo accounts; AI exposed them.",
}

def base(cat):
    return re.sub(r"\s*\(.*?\)\s*$", "", str(cat)).strip()

def attribute(cat, app):
    if (cat, app) in PROMPT_ASSOCIATED:
        return "Prompt-associated", PROMPT_ASSOCIATED[(cat, app)]
    if cat in PLATFORM_DEFAULT:
        p = "Bolt.new" if app[0] == "B" else "Lovable"
        n = int(prev.loc[cat, p]) if cat in prev.index else 0
        return "Platform default", f"Present in {n} apps on {p}; no team requested it; known stack default."
    return "AI implementation gap", AI_GAP_WHY.get(base(cat), "User specified the rule; app did not enforce it.")

df["origin"], df["evidence"] = zip(*df.apply(lambda r: attribute(r["cat"], r["app"]), axis=1))

# ---- 3. report ----
c = Counter(df["origin"]); tot = len(df)
for o in ["Platform default", "AI implementation gap", "Prompt-associated"]:
    print(f"{o}: {c.get(o,0)} ({100*c.get(o,0)/tot:.1f}%)")

In [101]:
OUT_DIR = "figures"

In [102]:
df["type"] = df["cat"]
vp = df
N_APPS = df["app"].nunique()

ORIGIN_COLOURS = {"Platform default": "#6C7A89",
                  "AI implementation gap": "#F4A261",
                  "Prompt-associated": "#E76F51",
                  "Not traceable": "#D9D9D9"}
ORIGIN_ORDER = list(ORIGIN_COLOURS)

In [103]:
# One colour per platform, used in every chart
BOLT_COLOUR = "#2B59C3"
LOVABLE_COLOUR = "#D1495B"

SEVERITY_ORDER = ["Critical", "High", "Medium", "Low"]
SEVERITY_COLOURS = {"Critical": "#7F1D1D", "High": "#E4572E",
                    "Medium": "#F3A712", "Low": "#9DB4C0"}

In [104]:
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

In [105]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    print("saved", name)

In [ ]:
print([t for t in vp["type"].unique() if "password" in t.lower()])

In [ ]:
print([a for a in vp["app"].unique() if "17" in a])

In [ ]:
print(vp[(vp["app"]=="L-T17") & (vp["type"].str.contains("password", case=False))][["app","type"]])

In [ ]:
print(vp["origin"].value_counts())

In [110]:
tot = vp["origin"].value_counts().reindex(ORIGIN_ORDER, fill_value=0)   # legend uses this
...
for o in ORIGIN_ORDER:
    share = tot[o] / tot.sum() * 100     # bar % uses this

In [ ]:
print(vp["origin"].value_counts())
print(ORIGIN_ORDER)

In [ ]:
print("origin counts:")
print(vp["origin"].value_counts())
print("ORIGIN_ORDER:", ORIGIN_ORDER)
print("tot:")
print(vp["origin"].value_counts().reindex(ORIGIN_ORDER, fill_value=0))

In [ ]:
print(tot)
print("prompt share:", tot["Prompt-associated"] / tot.sum() * 100)

In [ ]:
tot = vp["origin"].value_counts().reindex(ORIGIN_ORDER, fill_value=0)
tot

In [115]:
ORIGIN_COLOURS = {"Platform default": "#6C7A89",
                  "Prompt-associated": "#E76F51",
                  "AI implementation gap": "#F4A261",
                  "Not traceable": "#D9D9D9"}

In [ ]:
# ============================================================
# FIG R3-1  Origin of every verified vulnerability (no title; add on slide)
#   TOP: count per origin (bars sized by count, so the 1 is readable)
#   BOTTOM: each category, apps affected, split by origin
# ============================================================
T = pd.crosstab(vp["type"], vp["origin"]).reindex(columns=ORIGIN_ORDER, fill_value=0)
T["_main"] = T.idxmax(axis=1).map({o: i for i, o in enumerate(ORIGIN_ORDER)})
T["_tot"]  = T[ORIGIN_ORDER].sum(axis=1)
T = T.sort_values(["_main", "_tot"], ascending=[True, False])
tot = vp["origin"].value_counts().reindex(ORIGIN_ORDER, fill_value=0)
present = [o for o in ORIGIN_ORDER if tot[o] > 0]          # skip empty origins

fig, (ax0, ax) = plt.subplots(2, 1, figsize=(11, 0.42 * len(T) + 3.0),
                              gridspec_kw={"height_ratios": [1.0, len(T)]})

# TOP: one bar per origin, sized by count (NOT a 100% bar)
yp = list(range(len(present)))[::-1]
for o, y in zip(present, yp):
    ax0.barh(y, tot[o], color=ORIGIN_COLOURS[o], height=0.62)
    ax0.text(tot[o] + 1.5, y, f"{int(tot[o])}  ({tot[o] / tot.sum() * 100:.1f}%)",
             va="center", fontsize=10.5, fontweight="bold", color=ORIGIN_COLOURS[o])
ax0.set_yticks(yp); ax0.set_yticklabels(present, fontsize=10)
ax0.set_xlim(0, tot.max() * 1.3)
ax0.set_xlabel(f"Verified vulnerabilities (of {tot.sum()})", fontsize=9)
ax0.set_xticks([])
for s in ["top", "right", "bottom"]: ax0.spines[s].set_visible(False)
ax0.tick_params(length=0)

# BOTTOM: one bar per category, apps affected, split by origin
for y, (t, r) in enumerate(T.iterrows()):
    lft = 0
    for o in ORIGIN_ORDER:
        if r[o]:
            ax.barh(y, r[o], left=lft, color=ORIGIN_COLOURS[o], height=0.72)
            lft += r[o]
    ax.text(lft + 0.15, y, str(int(r["_tot"])), va="center", fontsize=9)
ax.set_yticks(range(len(T))); ax.set_yticklabels(T.index, fontsize=9.5)
ax.invert_yaxis()
ax.set_xlim(0, N_APPS + 1.5); ax.set_xticks(range(0, N_APPS + 1, 3))
ax.set_xlabel(f"Applications affected (of {N_APPS})")
ax.tick_params(axis="y", length=0)
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)

handles = [matplotlib.patches.Patch(color=ORIGIN_COLOURS[o], label=f"{o} ({tot[o]})") for o in present]
ax.legend(handles=handles, frameon=False, ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.12), fontsize=9)
save(fig, "fig_ro3_1_origin.png")

In [ ]:
print(prompts.columns.tolist())
print("B3 sum:", prompts["B3"].sum())

In [ ]:
b3 = prompts[prompts["B3"] == 1].copy()
b3["resp"] = np.where(b3["Suggested reply code"].astype(str).str.startswith("R8"),
                      "Complied without warning (R8)",
                      np.where(b3["Suggested reply code"].astype(str).str.startswith("R4"),
                               "Risk Communication (R4)", np.where(b3["Suggested reply code"].astype(str).str.startswith("R3"),
                                "Security Claim (R3)", "Not determinable from summary")))
# ---- T1: B3 weakening requests by AI response ----
T1 = (pd.crosstab(b3["Platform"], b3["resp"])
        .reindex(index=["Bolt", "Lovable"],
                 columns=["Complied without warning (R8)", "Risk Communication (R4)","Security Claim (R3)",
                          "Response not coded"], fill_value=0))
T1["Total requests"] = T1.sum(axis=1)
T1.loc["Total"] = T1.sum()

print("T1  Security-weakening requests (B3) and AI response\n")
print(T1, "\n")

In [ ]:
# F2: R3 security claims
print(f"\n=== F2. AI security claims (R3), over {sec} security-relevant prompts ===")
print(f"  R3 instances: {len(r3)} "
      f"(Bolt {sum(1 for x in r3 if x[0]=='Bolt')} / Lovable {sum(1 for x in r3 if x[0]=='Lovable')})")
teams = sorted({x[1] for x in r3})
print(f"  Spread across {len(teams)} application(s): {teams}")
for p, i, rc, t in r3: print(f"    {p:8} {i} [{rc}] {t}")

# F3: R4 risk communication
print(f"\n=== F3. AI risk communication (R4), over {sec} security-relevant prompts ===")
print(f"  R4 instances: {len(r4)}")
if not r4:
    print("  (none coded yet - add 'R4' to the Reply codes cell where the AI warned of a risk)")
for p, i, rc, t in r4: print(f"    {p:8} {i} [{rc}] {t}")

In [ ]:
# ============================================================
# FIG R3-3  Trace examples: prompt -> AI reply -> test result
# ============================================================
TRACES = [
    # 1. PROMPT-ASSOCIATED — the one real case (your clearest example)
    ("L-T17 · Lovable  (Prompt-associated)", LOVABLE_COLOUR,
     "Prompt #16 (B3)\n\"Do not block any sign up because of\nweak password, allow all types of password\"",
     "AI reply\nComplied — leaked-password protection\nturned off, no warning given.",
     "Test result\nLeaked-password check disabled\n(verified, Low)", "#C0392B"),

    # 2. AI IMPLEMENTATION GAP — user specified the rule, AI even CLAIMED it worked
    ("L-T20 · Lovable  (AI implementation gap)", LOVABLE_COLOUR,
     "Prompt #1\n\"Owner... only role that can give\nfinal approval to finalize invoices.\"",
     "AI reply (R3, prompt #10)\n\"Final approval must come from the\nsigned-in Owner — working as designed.\"",
     "Test result\nA lower role could approve\n(verified, Medium) — claim was false", "#C0392B"),

    # 3. AI IMPLEMENTATION GAP — the strongest: user wrote the RLS rule, app still failed
    ("L-T12 · Lovable  (AI implementation gap)", LOVABLE_COLOUR,
     "Prompt #2\n\"Staff: Cannot approve or finalize invoices.\"\n+ RLS pattern: business_id = auth.jwt()\u2192business_id",
     "AI reply\nBuilt the app; claimed role-based\naccess control was enforced.",
     "Test result\nStaff escalated to owner & approved\nan invoice (verified, High)", "#C0392B"),
]
fig, ax = plt.subplots(figsize=(13, 2.0 * len(TRACES)))
ax.set_xlim(0, 13)
ax.set_ylim(0, 2.3 * len(TRACES))
ax.axis("off")
for i, (label, pc, p, r, t, tc) in enumerate(TRACES):
    y = 2.3 * (len(TRACES) - 1 - i) + 0.35
    ax.text(0.05, y + 1.75, label, fontsize=11, fontweight="bold", color=pc)
    for x, txt, edge, face in [(0.05, p, pc, "#F8FAFC"), (4.45, r, "#6B7280", "#F8FAFC"), (8.85, t, tc, "#FFFFFF")]:
        box = matplotlib.patches.FancyBboxPatch((x, y + 0.3), 4.0, 1.2, boxstyle="round,pad=0.05,rounding_size=0.12",
                                                linewidth=2, edgecolor=edge, facecolor=face)
        ax.add_patch(box)
        head, body = txt.split("\n", 1)
        ax.text(x + 0.15, y + 1.3, head, fontsize=10, fontweight="bold", va="top", color="#111827")
        ax.text(x + 0.15, y + 0.95, body, fontsize=9.3, va="top", color="#374151")
    for x in (4.08, 8.48):
        ax.annotate("", xy=(x + 0.33, y + 0.75), xytext=(x, y + 0.75),
                    arrowprops=dict(arrowstyle="-|>", color="#6B7280", lw=1.8))
save(fig, "fig_ro3_3_traces.png")